## GSAT trend patterns

In [ ]:
# In[1]:
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
# %%
# define function
import src.SAT_function_Obs_Fingerprint as data_process
import src.Data_Preprocess as preprocess
from src.Statistic_cal import pattern_rmse

In [ ]:
# import src.slurm_cluster as scluster
# client, scluster = scluster.init_dask_slurm_cluster()

In [ ]:
# Input the observational trend
variable_name = ['2013-2022', '1993-2022', '1963-2022', '1979-2022']

# Input the Observational forced trend (wrt MMEM GSAT)
dir_internal_input = '/work/mh0033/m301036/OBS_LPS_revision/docs/data/FIG3/OBS_ICV_std/concatenate/'
trend_length=['10', '30', '60', '44']
HadCRUT5_annual_internal_trend_da = {}

for interval, L in zip(variable_name, trend_length):
    HadCRUT5_annual_internal_trend_da[interval] = xr.open_dataset(dir_internal_input + 'OBS_ICV_MK_trend_STD_1950_2022_sliding.nc')

In [ ]:
# The OBS field has an extra trend_length dim; pick the diagonal so period[i] uses trend_length[i]
obs_ds = HadCRUT5_annual_internal_trend_da[interval]
obs_raw = obs_ds["icv_trend_std"] if "icv_trend_std" in obs_ds else next(iter(obs_ds.data_vars.values()))
if "trend_length" in obs_raw.dims:
    n_match = min(obs_raw.sizes["period"], obs_raw.sizes["trend_length"])
    idx = xr.DataArray(np.arange(n_match), dims=["period"])
    obs_icv = obs_raw.isel(period=idx, trend_length=idx)
    obs_icv = obs_icv.drop_vars([v for v in obs_icv.coords if v == "trend_length"], errors="ignore")
else:
    obs_icv = obs_raw

In [ ]:
obs_icv

In [ ]:
# Input the MMEM annual trend
dir_model_in = '/work/mh0033/m301036/OBS_LPS_revision/docs/data/FIG3/{model}/SMILE_internal/'
model_name = ["MMLE", "MIROC6", "MPI_ESM", "ACCESS", "EC_Earth3", "IPSL_CM6A", "CESM2", "CanESM5"]

LE_internal_trend_da = {}
for model in model_name:
    LE_internal_trend_da[model] = {}
    for interval in variable_name:
        if model == 'MMLE':
            LE_internal_trend_da[model][interval] = xr.open_dataset(dir_model_in.format(model=model) + '{model}_internal_trend_std_1950-2022_sliding.nc'.format(model=model)).trend.sel(period=interval)
        else:
            LE_internal_trend_da[model][interval] = xr.open_dataset(dir_model_in.format(model=model) + '{model}_SMILE_noise_trend_std_sliding_1950_2022.nc'.format(model=model)).noise_trend_std.sel(period=interval)

In [ ]:
# pattern difference between LE forced trend and obs trend (align first)
LE_internal_trend_diff_da = {}
for model in model_name:
    LE_internal_trend_diff_da[model] = {}
    for period in variable_name:
        obs_field = obs_icv.sel(period=period)
        model_field = LE_internal_trend_da[model][period]
        obs_aln, model_aln = xr.align(obs_field, model_field, join="inner")
        LE_internal_trend_diff_da[model][period] = model_aln - obs_aln


In [ ]:
LE_internal_trend_diff_da['MIROC6']["1963-2022"]

In [ ]:
LE_internal_trend_da['MMLE']

### The amplitude ratio is 
$\text{bias\_fraction} = \dfrac{A_{\text{bias}}}{A_{\text{obs}}}
 = \dfrac{\mathrm{RMS}(F_{\text{ENS}} - F_{\text{OBS}})}{\mathrm{RMS}(F_{\text{OBS}})}$.

In [ ]:
def area_weighted_rms(field, lat):
    w = np.cos(np.deg2rad(lat))
    w = w / w.mean()
    w2 = w.broadcast_like(field)
    wsum = w2.sum(dim=("lat", "lon"))
    return np.sqrt((w2 * field**2).sum(dim=("lat", "lon")) / wsum)


In [ ]:
LE_internal_trend_diff_amplitude_da = {}
for model in model_name:
    LE_internal_trend_diff_amplitude_da[model] = {}
    for interval in variable_name:
        LE_internal_trend_diff_amplitude_da[model][interval] = area_weighted_rms(LE_internal_trend_diff_da[model][interval], LE_internal_trend_diff_da[model][interval].lat)

In [ ]:
LE_internal_trend_diff_amplitude_da['CESM2']


In [ ]:
# calculate the observed internal trend amplitude
HadCRUT5_internal_trend_amplitude_da = {}
for interval in variable_name:
    # select by period label rather than item access to avoid KeyError
    obs_field = obs_icv.sel(period=interval)
    HadCRUT5_internal_trend_amplitude_da[interval] = area_weighted_rms(obs_field, obs_field.lat)

In [ ]:
HadCRUT5_internal_trend_amplitude_da['2013-2022']

In [ ]:
# calculate the bias fraction without the trend_length dimension
LE_internal_trend_diff_fraction_da = {}
for model in model_name:
    LE_internal_trend_diff_fraction_da[model] = {}
    for interval in variable_name:
        num = LE_internal_trend_diff_amplitude_da[model][interval]
        den = HadCRUT5_internal_trend_amplitude_da[interval]

        num_scalar = num.mean(dim="trend_length", skipna=True) if "trend_length" in num.dims else num
        den_scalar = den.mean(dim="trend_length", skipna=True) if "trend_length" in den.dims else den

        LE_internal_trend_diff_fraction_da[model][interval] = (num_scalar / den_scalar) * 100

In [ ]:
LE_internal_trend_diff_fraction_da["CESM2"]

### Plotting with the Robinson Projections

In [ ]:
import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import matplotlib.ticker as mticker
import cartopy.feature as cfeature
import cartopy.mpl.ticker as cticker
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
import matplotlib.gridspec as gridspec
import matplotlib as mpl
import seaborn as sns
from matplotlib.colors import ListedColormap
from matplotlib.colors import BoundaryNorm, ListedColormap
from src.plot_func import *
set_science_advances_style(column='double')  # or 'single'

def plot_trend_with_significance(trend_data, lats, lons, p_values, GMST_p_values=None, levels=None, extend=None, cmap=None, 
                                 title="", ax=None, show_xticks=False, show_yticks=False):
    """
    Plot the trend spatial pattern using Robinson projection with significance overlaid.

    Parameters:
    - trend_data: 2D numpy array with the trend values.
    - lats, lons: 1D arrays of latitudes and longitudes.
    - p_values: 2D array with p-values for each grid point.
    - GMST_p_values: 2D array with GMST p-values for each grid point.
    - title: Title for the plot.
    - ax: Existing axis to plot on. If None, a new axis will be created.
    - show_xticks, show_yticks: Boolean flags to show x and y axis ticks.
    
    Returns:
    - contour_obj: The contour object from the plot.
    """

    # Create a new figure/axis if none is provided
    if ax is None:
        fig, ax = plt.subplots(figsize=(20, 15), subplot_kw={'projection': ccrs.Robinson()})
        ax.set_global()
  
    insignificance_mask = p_values >= 0.05
    
    # Plotting
    # contour_obj = ax.pcolormesh(lons, lats, trend_data,  cmap='RdBu_r',vmin=-5.0, vmax=5.0, transform=ccrs.PlateCarree(central_longitude=180), shading='auto')
    contour_obj = ax.contourf(lons, lats, trend_data, levels=levels, extend=extend, cmap=cmap, transform=ccrs.PlateCarree(central_longitude=0))

    # Plot significance masks with different hatches
    ax.contourf(lons, lats, insignificance_mask, levels=[0, 0.05, 1.0],hatches=[None,'///'], colors='none', transform=ccrs.PlateCarree())

    ax.coastlines(resolution='110m')
    gl = ax.gridlines(draw_labels=True, dms=True, x_inline=False, y_inline=False,
                      color='gray', alpha=0.35, linestyle='--')

    # Disable labels on the top and right of the plot
    gl.top_labels = False
    gl.right_labels = False

    # Enable labels on the bottom and left of the plot
    gl.bottom_labels = show_xticks
    gl.left_labels = show_yticks
    gl.xformatter = cticker.LongitudeFormatter()
    gl.yformatter = cticker.LatitudeFormatter()
    gl.xlabel_style = {'size': 15}
    gl.ylabel_style = {'size': 15}
    
    if show_xticks:
        gl.bottom_labels = True
    if show_yticks:
        gl.left_labels = True
    
    ax.set_title(title, loc='center', fontsize=18, pad=5.0)

    return contour_obj

In [ ]:
def plot_trend(trend_data, lats, lons, levels=None, extend=None, cmap=None, 
                                 title="", ax=None, show_xticks=False, show_yticks=False):
    """
    Plot the trend spatial pattern using Robinson projection with significance overlaid.

    Parameters:
    - trend_data: 2D numpy array with the trend values.
    - lats, lons: 1D arrays of latitudes and longitudes.
    - p_values: 2D array with p-values for each grid point.
    - GMST_p_values: 2D array with GMST p-values for each grid point.
    - title: Title for the plot.
    - ax: Existing axis to plot on. If None, a new axis will be created.
    - show_xticks, show_yticks: Boolean flags to show x and y axis ticks.
    
    Returns:
    - contour_obj: The contour object from the plot.
    """

    # Create a new figure/axis if none is provided
    if ax is None:
        fig, ax = plt.subplots(figsize=(20, 15), subplot_kw={'projection': ccrs.Robinson()})
        ax.set_global()
  
    contour_obj = ax.contourf(lons, lats, trend_data, levels=levels, extend=extend, cmap=cmap, transform=ccrs.PlateCarree(central_longitude=0))

    ax.coastlines(resolution='110m')
    gl = ax.gridlines(draw_labels=True, dms=True, x_inline=False, y_inline=False,
                      color='gray', alpha=0.35, linestyle='--')

    # Disable labels on the top and right of the plot
    gl.top_labels = False
    gl.right_labels = False

    # Enable labels on the bottom and left of the plot
    gl.bottom_labels = show_xticks
    gl.left_labels = show_yticks
    gl.xformatter = cticker.LongitudeFormatter()
    gl.yformatter = cticker.LatitudeFormatter()
    gl.xlabel_style = {'size': 15}
    gl.ylabel_style = {'size': 15}
    
    if show_xticks:
        gl.bottom_labels = True
    if show_yticks:
        gl.left_labels = True
    
    ax.set_title(title, loc='center', fontsize=18, pad=5.0)

    return contour_obj

In [ ]:
# define an asymmetric colormap
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.colors import BoundaryNorm
import cartopy.util as cutil
import seaborn as sns
import matplotlib.colors as mcolors
import palettable

cmap=mcolors.ListedColormap(palettable.cmocean.diverging.Balance_20.mpl_colors)

In [ ]:
LE_internal_trend_diff_da['MIROC6']['1993-2022']

### Plot the Original, internal, MMEM trend patterns

In [ ]:
# arange data into a list arrocding to the variable name
trend_2013_2022 = {"HadCRUT5": obs_icv.sel(period='2013-2022'),
            "MMLE":LE_internal_trend_diff_da['MMLE']['2013-2022'],
           "MIROC6":LE_internal_trend_diff_da['MIROC6']['2013-2022'],
           "MPI_ESM":LE_internal_trend_diff_da['MPI_ESM']['2013-2022'],
           "ACCESS":LE_internal_trend_diff_da['ACCESS']['2013-2022'], 
           "EC_Earth3":LE_internal_trend_diff_da['EC_Earth3']['2013-2022'],
           "IPSL_CM6A":LE_internal_trend_diff_da['IPSL_CM6A']['2013-2022'],
           "CESM2":LE_internal_trend_diff_da['CESM2']['2013-2022'],
           "CanESM5":LE_internal_trend_diff_da['CanESM5']['2013-2022']
            }

trend_1993_2022 = {"HadCRUT5": obs_icv.sel(period='1993-2022'),
            "MMLE":LE_internal_trend_diff_da['MMLE']['1993-2022'],
           "MIROC6":LE_internal_trend_diff_da['MIROC6']['1993-2022'],
           "MPI_ESM":LE_internal_trend_diff_da['MPI_ESM']['1993-2022'],
           "ACCESS":LE_internal_trend_diff_da['ACCESS']['1993-2022'], 
           "EC_Earth3":LE_internal_trend_diff_da['EC_Earth3']['1993-2022'],
           "IPSL_CM6A":LE_internal_trend_diff_da['IPSL_CM6A']['1993-2022'],
           "CESM2":LE_internal_trend_diff_da['CESM2']['1993-2022'],
           "CanESM5":LE_internal_trend_diff_da['CanESM5']['1993-2022']
            }

trend_1963_2022 = {"HadCRUT5": obs_icv.sel(period='1963-2022'),
            "MMLE":LE_internal_trend_diff_da['MMLE']['1963-2022'],
           "MIROC6":LE_internal_trend_diff_da['MIROC6']['1963-2022'],
           "MPI_ESM":LE_internal_trend_diff_da['MPI_ESM']['1963-2022'],
           "ACCESS":LE_internal_trend_diff_da['ACCESS']['1963-2022'], 
           "EC_Earth3":LE_internal_trend_diff_da['EC_Earth3']['1963-2022'],
           "IPSL_CM6A":LE_internal_trend_diff_da['IPSL_CM6A']['1963-2022'],
           "CESM2":LE_internal_trend_diff_da['CESM2']['1963-2022'],
           "CanESM5":LE_internal_trend_diff_da['CanESM5']['1963-2022']   
            }

trend_1979_2022 = {"HadCRUT5": obs_icv.sel(period='1979-2022'),
            "MMLE":LE_internal_trend_diff_da['MMLE']['1979-2022'],
           "MIROC6":LE_internal_trend_diff_da['MIROC6']['1979-2022'],
           "MPI_ESM":LE_internal_trend_diff_da['MPI_ESM']['1979-2022'],
           "ACCESS":LE_internal_trend_diff_da['ACCESS']['1979-2022'], 
           "EC_Earth3":LE_internal_trend_diff_da['EC_Earth3']['1979-2022'],
           "IPSL_CM6A":LE_internal_trend_diff_da['IPSL_CM6A']['1979-2022'],
           "CESM2":LE_internal_trend_diff_da['CESM2']['1979-2022'],
           "CanESM5":LE_internal_trend_diff_da['CanESM5']['1979-2022']   
            }

In [ ]:
trend_2013_2022

In [ ]:
# define an asymmetric colormap
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.colors import BoundaryNorm
import cartopy.util as cutil
import seaborn as sns
import matplotlib.colors as mcolors
import palettable

In [ ]:
# # pattern correlation betwenn observed forced pattern vs. Model simulated forced pattern
# import scipy.stats as stats

# trend_pattern_correlation_10yr = []

# for i in range(8):
#     trend_pattern_correlation_10yr.append(stats.pearsonr(trend_2013_2022['HadCRUT5'].values.flatten(), trend_2013_2022[model_name[i]].values.flatten())[0])

# trend_pattern_correlation_10yr 

In [ ]:
# trend_pattern_correlation_30yr = []
# for i in range(len(model_name)):
#     trend_pattern_correlation_30yr.append(stats.pearsonr(trend_1993_2022['HadCRUT5'].values.flatten(), trend_1993_2022[model_name[i]].values.flatten())[0])
# trend_pattern_correlation_30yr

In [ ]:
# trend_pattern_correlation_44yr = []
# for i in range(len(model_name)):
#     trend_pattern_correlation_44yr.append(stats.pearsonr(trend_1979_2022['HadCRUT5'].values.flatten(), trend_1979_2022[model_name[i]].values.flatten())[0])
# trend_pattern_correlation_44yr

In [ ]:
# trend_pattern_correlation_60yr = []
# for i in range(len(model_name)):
#     trend_pattern_correlation_60yr.append(stats.pearsonr(trend_1963_2022['HadCRUT5'].values.flatten(), trend_1963_2022[model_name[i]].values.flatten())[0])
# trend_pattern_correlation_60yr

In [ ]:
# trend_pattern_correlation_10yr

In [ ]:
# # save the pattern correlation from the second to the last, which corresponds to the MMEM, CanESM5, IPSL, EC-Earth3, ACCESS, MPI-ESM, MIROC6
# with open('pattern_correlations_noise_trend_std_model_vs_Obs.txt', 'w') as file:
#     file.write('10-year Trend Pattern Correlations:\n')
#     for correlation in trend_pattern_correlation_10yr:
#         file.write(f"{correlation}\n")

#     file.write('\n30-year Trend Pattern Correlations:\n')
#     for correlation in trend_pattern_correlation_30yr:
#         file.write(f"{correlation}\n")
        
#     file.write('\n44-year Trend Pattern Correlations:\n')
#     for correlation in trend_pattern_correlation_44yr:
#         file.write(f"{correlation}\n")

#     file.write('\n60-year Trend Pattern Correlations:\n')
#     for correlation in trend_pattern_correlation_60yr:
#         file.write(f"{correlation}\n")


In [ ]:
# Plotting
lat = trend_2013_2022['HadCRUT5'].lat
lon = trend_2013_2022['HadCRUT5'].lon
lat, lon

titles_rows = ["HadCRUT5", "MMLE", "MIROC6", "MPI-ESM1.2-LR", "ACCESS-ESM1.5", "EC-Earth3", "IPSL-CM6A-LR", "CESM2", "CanESM5"]
rows_label = ["A", "B", "C", "D", "E", "F", "G", "H", "I"]
titles_columns = ["2013-2022 (10yr)", "1993-2022 (30yr)", "1963-2022 (60yr)"]

import cartopy.util as cutil
import matplotlib.colors as mcolors
import palettable

periods = ["10yr", "30yr", "60yr"]
variable_name = ["HadCRUT5", "MMLE", "MIROC6", "MPI_ESM", "ACCESS", "EC_Earth3", "IPSL_CM6A", "CESM2", "CanESM5"]

intervals = np.arange(0.0, 1.05, 0.05)
intervals_diff = np.arange(-0.25, 0.275, 0.025)

cmap = mcolors.ListedColormap(palettable.cmocean.sequential.Amp_20.mpl_colors)
cmap_diff = "RdBu_r"
extend_obs = 'max'
extend_diff = 'both'

panel_fs = 10
row_label_fs = 8
corr_fs = 9
cbar_fs = 8

fig = plt.figure(figsize=(20, 35))
gs = gridspec.GridSpec(9, 3, figure=fig, wspace=0.05, hspace=0.05)

axes = {}
obs_contour = None
diff_contour = None

for i, var in enumerate(variable_name):
    for j, period in enumerate(periods):
        is_left = j == 0
        is_bottom_row = i >= 8

        ax = plt.subplot(gs[i, j], projection=ccrs.Robinson(180))
        ax.set_global()
        axes[i, j] = ax
        if j == 0:
            trend_data = trend_2013_2022[var]
        elif j == 1:
            trend_data = trend_1993_2022[var]
        else:
            trend_data = trend_1963_2022[var]

        axis_lon = trend_data.get_axis_num("lon")
        trend_with_cyclic, lons_cyclic = cutil.add_cyclic_point(trend_data, coord=lon, axis=axis_lon)

        if i == 0:
            contour = plot_data(
                trend_with_cyclic, lat, lons_cyclic,
                levels=intervals, extend=extend_obs,
                cmap=cmap, title=" ", ax=ax,
                show_xticks=is_bottom_row, show_yticks=is_left
            )
            if obs_contour is None:
                obs_contour = contour
        else:
            contour = plot_data(
                trend_with_cyclic, lat, lons_cyclic,
                levels=intervals_diff, extend=extend_diff,
                cmap=cmap_diff, title=" ", ax=ax,
                show_xticks=is_bottom_row, show_yticks=is_left
            )
            if diff_contour is None:
                diff_contour = contour

for i, var in enumerate(variable_name):
    axes[i, 0].text(-0.2, 0.5, titles_rows[i], va='center', ha='center', rotation=90,
                    fontsize=20, transform=axes[i, 0].transAxes)
    axes[i, 0].text(-0.08, 1.05, rows_label[i], va='bottom', ha='right', rotation='horizontal',
                    fontsize=24, fontweight='bold', transform=axes[i, 0].transAxes)

for j in range(3):
    axes[0, j].text(0.5, 1.05, titles_columns[j], va='bottom', ha='center', rotation='horizontal',
                    fontsize=20, transform=axes[0, j].transAxes)

model_name = ["MMLE", "MIROC6", "MPI_ESM", "ACCESS", "EC_Earth3", "IPSL_CM6A", "CESM2", "CanESM5"]
for i, model in enumerate(model_name):
    axes[i + 1, 0].text(0.35, 1.0, f"bias_frac: {LE_internal_trend_diff_fraction_da[model]['2013-2022']:.2f}%", va='bottom', ha='left', fontsize=20, transform=axes[i + 1, 0].transAxes)
    axes[i + 1, 1].text(0.35, 1.0, f"bias_frac: {LE_internal_trend_diff_fraction_da[model]['1993-2022']:.2f}%", va='bottom', ha='left', fontsize=20, transform=axes[i + 1, 1].transAxes)
    axes[i + 1, 2].text(0.35, 1.0, f"bias_frac: {LE_internal_trend_diff_fraction_da[model]['1963-2022']:.2f}%", va='bottom', ha='left', fontsize=20, transform=axes[i + 1, 2].transAxes)

# Colorbar for observations
cbar_ax_obs = fig.add_axes([0.1, 0.08, 0.35, 0.01])
cbar_obs = plt.colorbar(obs_contour, cax=cbar_ax_obs, orientation='horizontal', extend=extend_obs)
cbar_obs.ax.tick_params(labelsize=16)
cbar_obs.set_label('Observation SAT trend Stddev. (°C per decade)', fontsize=16)
cbar_obs.ax.tick_params(direction='out', length=8, width=2)
cbar_obs.ax.minorticks_off()

# Colorbar for model differences
cbar_ax_diff = fig.add_axes([0.55, 0.08, 0.35, 0.01])
cbar_diff = plt.colorbar(diff_contour, cax=cbar_ax_diff, orientation='horizontal', extend=extend_diff)
cbar_diff.ax.tick_params(labelsize=16)
cbar_diff.set_label('Model bias SAT trend Stddev. difference (°C per decade)', fontsize=16)
cbar_diff.ax.tick_params(direction='out', length=8, width=2)
cbar_diff.ax.minorticks_off()

figure_output = '/work/mh0033/m301036/OBS_LPS_revision/docs/Figs/FIG_S7_S8/'
for ext in ("png", "pdf",):
    fig.savefig(figure_output + f"FIGURE_S8_OBS_LE_ICV_trend_STD_diff_bias_fraction.{ext}", format=ext, dpi=300, bbox_inches='tight')

plt.show()

In [ ]:
# Plotting
lat = trend_2013_2022['HadCRUT5'].lat
lon = trend_2013_2022['HadCRUT5'].lon
lat, lon 

titles_rows = ["HadCRUT5", "MMLE", "MIROC6", "MPI-ESM1.2-LR","ACCESS-ESM1.5", "EC-Earth3", "IPSL-CM6A-LR", "CESM2", "CanESM5"]
# rows_label = ["a", "b", "c", "d", "e", "f", "g"]
rows_label = ["A", "B", "C", "D", "E", "F", "G", "H", "I"]
titles_columns = ["2013-2022 (10yr)", "1993-2022 (30yr)", "1979-2022 (44yr)", "1963-2022 (60yr)"]
# ["10-year (2013-2022)", "30-year (1993-2022)", "60-year (1963-2022)"]
import cartopy.util as cutil
import seaborn as sns
import matplotlib.colors as mcolors
import palettable

periods = ["10yr", "30yr", "44yr", "60yr"]
variable_name = ["HadCRUT5", "MMLE", "MIROC6", "MPI_ESM", "ACCESS", "EC_Earth3", "IPSL_CM6A", "CESM2", "CanESM5"]

intervals = np.arange(0.0, 1.05, 0.05)

cmap = mcolors.ListedColormap(palettable.cmocean.sequential.Amp_20.mpl_colors)
extend = 'both'

panel_fs = 10
row_label_fs = 8
corr_fs = 9
cbar_fs = 8
# set_science_advances_style(
fig = plt.figure(figsize=(25, 30)) 
gs = gridspec.GridSpec(9, 4, figure=fig, width_ratios=[1,1,1,1], wspace=0.08, hspace=0.015)

# Create a 8x3 grid of subplots
axes = {}
for i, var in enumerate(variable_name):
    for j, period in enumerate(periods):
        is_left = j == 0
        is_bottom_row = i >= 8
        
        ax = plt.subplot(gs[i, j], projection=ccrs.Robinson(180))
        ax.set_global()
        axes[i, j] = ax
                       
        if j == 0:
        # Add cyclic points
            trend_data = trend_2013_2022[variable_name[i]]
            trend_with_cyclic, lons_cyclic = cutil.add_cyclic_point(trend_data, coord=lon)
                    
            contour_obj1 = plot_trend(trend_with_cyclic, lat, lons_cyclic, 
                                            levels=intervals, extend = 'max',
                                            cmap=cmap, title=" ", ax=ax, 
                                            show_xticks = is_bottom_row, show_yticks = is_left)
        elif j == 1:
            trend_data = trend_1993_2022[variable_name[i]]
            trend_with_cyclic, lons_cyclic = cutil.add_cyclic_point(trend_data, coord=lon)
                    
            contour_obj2 = plot_trend(trend_with_cyclic, lat, lons_cyclic, 
                                            levels=intervals, extend = 'max',
                                            cmap=cmap, title=" ", ax=ax, 
                                            show_xticks = is_bottom_row, show_yticks = is_left)
        elif j == 2:
            trend_data = trend_1979_2022[variable_name[i]]
            trend_with_cyclic, lons_cyclic = cutil.add_cyclic_point(trend_data, coord=lon)
                    
            contour_obj3 = plot_trend(trend_with_cyclic, lat, lons_cyclic, 
                                            levels=intervals, extend = 'max',
                                            cmap=cmap, title=" ", ax=ax, 
                                            show_xticks = is_bottom_row, show_yticks = is_left)
        else:
            trend_data = trend_1963_2022[variable_name[i]]
            trend_with_cyclic, lons_cyclic = cutil.add_cyclic_point(trend_data, coord=lon)
      
            contour_obj4 = plot_trend(trend_with_cyclic, lat, lons_cyclic,
                                    levels=intervals, extend = extend,
                                    cmap=cmap, title=" ", ax=ax, 
                                    show_xticks = is_bottom_row, show_yticks = is_left)

# add the title for each row
for i, var in enumerate(variable_name):
    axes[i, 0].text(-0.2, 0.5, titles_rows[i], va='center', ha='center', rotation=90, 
                    fontsize=20,transform=axes[i, 0].transAxes)
    axes[i, 0].text(-0.08, 1.05, rows_label[i], va='bottom', ha='right', rotation='horizontal',
                    fontsize=24, fontweight='bold', transform=axes[i, 0].transAxes)

# add the title for each column
for j in range(4):
    axes[0,j].text(0.5, 1.05, titles_columns[j], va='bottom', ha='center', rotation='horizontal', 
                   fontsize=20,transform=axes[0, j].transAxes)

# Add the pattern correlation text to the second to last rows of each column subplot
for i in np.arange(1,9,1):
    axes[i, 0].text(0.65, 1.0, f"corr: {trend_pattern_correlation_10yr[i-1]:.2f}", va='bottom', ha='left', fontsize=20,transform=axes[i, 0].transAxes)
    axes[i, 1].text(0.65, 1.0, f"corr: {trend_pattern_correlation_30yr[i-1]:.2f}", va='bottom', ha='left', fontsize=20,transform=axes[i, 1].transAxes)
    axes[i, 2].text(0.65, 1.0, f"corr: {trend_pattern_correlation_44yr[i-1]:.2f}", va='bottom', ha='left', fontsize=20,transform=axes[i, 2].transAxes)
    axes[i, 3].text(0.65, 1.0, f"corr: {trend_pattern_correlation_60yr[i-1]:.2f}", va='bottom', ha='left', fontsize=20,transform=axes[i, 3].transAxes)

# Add horizontal colorbars
cbar_ax = fig.add_axes([0.25, 0.08, 0.5, 0.01])
cbar = plt.colorbar(contour_obj1, cax=cbar_ax, orientation='horizontal', extend=extend)
cbar.ax.tick_params(labelsize=16)
cbar.set_label('SAT trend Stddev.(°C per decade)', fontsize=18)
cbar.ax.tick_params(direction='out', length=8, width=2)  
# remove the minor ticks from the bar labels
cbar.ax.minorticks_off()

# plt.figure(constrained_layout=True)
figure_output = '/work/mh0033/m301036/OBS_LPS_revision/docs/Figs/FIG_S7_S8/'
for ext in ("png", "pdf",):
    fig.savefig(figure_output+f"FIGURE_S8_OBS_LE_ICV_trend_STD_comparison_Satellite.{ext}", format=ext, dpi=300, bbox_inches='tight')

plt.show()